# Social E-commerce Purchase Prediction & Seller Recommendation System## Project Overview**Goal**: Build a data-driven recommendation system for sellers to improve product purchase rates.**Approach**:1. EDA & Feature Engineering2. Multi-model comparison (Logistic Regression / Random Forest / XGBoost)3. Model Interpretability (SHAP + Partial Dependence Plots + Counterfactual Analysis)4. K-Means User Clustering5. Actionable Seller Recommendations**Key Design Decisions**:- Drop `user_id` and `item_id` (unreliable)- Drop engagement features (`like_num`, `comment_num`, `share_num`, `collect_num`, `interaction_rate`, `social_influence`) to avoid data leakage- Recommendations are only based on **seller-controllable features**

---## 1. Data Loading & Preprocessing

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport warningsplt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']plt.rcParams['axes.unicode_minus'] = Falsewarnings.filterwarnings('ignore')# Load datadf_raw = pd.read_csv('social_ecommerce_data.csv')print(f'Dataset shape: {df_raw.shape}')df_raw.head()

In [ ]:
# Drop unreliable ID columns and engagement/leakage featuresdrop_cols = [    'user_id', 'item_id',    'like_num', 'comment_num', 'share_num', 'collect_num',    'interaction_rate', 'social_influence']df = df_raw.drop(columns=drop_cols)print(f'After dropping: {df.shape}')print(f'\nRemaining columns ({len(df.columns)}):')for col in df.columns:    print(f'  {col}: {df[col].dtype}')

In [ ]:
# Check missing values and basic statsprint('Missing values:')print(df.isnull().sum())print(f'\nLabel distribution:')print(df['label'].value_counts())print(f'\nPurchase rate: {df["label"].mean():.3f}')

In [ ]:
df.describe()

In [ ]:
# Define feature groups for later useSELLER_CONTROLLABLE = ['title_length', 'title_emo_score', 'img_count', 'has_video', 'price', 'discount_rate']MARKETING_FEATURES = ['coupon_received']USER_CONTEXT = ['age', 'gender', 'user_level', 'purchase_freq', 'total_spend', 'register_days', 'follow_num', 'fans_num']BEHAVIOR_FEATURES = ['is_follow_author', 'add2cart', 'coupon_used', 'pv_count', 'last_click_gap', 'purchase_intent', 'freshness_score']CATEGORY_COL = 'category'TARGET = 'label'print('Feature groups:')print(f'  Seller controllable: {len(SELLER_CONTROLLABLE)}')print(f'  Marketing:           {len(MARKETING_FEATURES)}')print(f'  User context:        {len(USER_CONTEXT)}')print(f'  Behavior:            {len(BEHAVIOR_FEATURES)}')print(f'  Category:            1')print(f'  Target:              1')

---## 2. Exploratory Data Analysis (EDA)

In [ ]:
# 2.1 Label distributionfig, axes = plt.subplots(1, 2, figsize=(12, 4))labels = ['Not Purchased (0)', 'Purchased (1)']counts = df['label'].value_counts().sort_index()colors = ['#e74c3c', '#2ecc71']axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5)for i, v in enumerate(counts.values):    axes[0].text(i, v + 500, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')axes[0].set_title('Overall Purchase Distribution', fontsize=13, fontweight='bold')axes[0].set_ylabel('Count')cat_rate = df.groupby('category')['label'].mean().sort_values(ascending=False)bars = axes[1].bar(cat_rate.index, cat_rate.values, color='#3498db', edgecolor='white', linewidth=1.5)for bar, v in zip(bars, cat_rate.values):    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.1%}', ha='center', fontweight='bold')axes[1].set_title('Purchase Rate by Category', fontsize=13, fontweight='bold')axes[1].set_ylabel('Purchase Rate')axes[1].tick_params(axis='x', rotation=15)plt.tight_layout()plt.show()

In [ ]:
# 2.2 Seller-controllable features vs purchase ratefig, axes = plt.subplots(2, 3, figsize=(16, 10))for idx, feat in enumerate(SELLER_CONTROLLABLE):    ax = axes[idx // 3][idx % 3]    if feat in ['has_video']:        rate = df.groupby(feat)['label'].mean()        bars = ax.bar(rate.index.astype(str), rate.values, color=['#e74c3c', '#2ecc71'], edgecolor='white')        for bar, v in zip(bars, rate.values):            ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.1%}', ha='center', fontweight='bold')        ax.set_xlabel(feat)        ax.set_ylabel('Purchase Rate')    else:        ax.hist(df[df['label']==0][feat], bins=30, alpha=0.6, label='Not Purchased', color='#e74c3c', density=True)        ax.hist(df[df['label']==1][feat], bins=30, alpha=0.6, label='Purchased', color='#2ecc71', density=True)        ax.set_xlabel(feat)        ax.set_ylabel('Density')        ax.legend(fontsize=8)    ax.set_title(f'{feat} vs Purchase', fontsize=11, fontweight='bold')plt.suptitle('Seller-Controllable Features vs Purchase', fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.show()

In [ ]:
# 2.3 User demographic features vs purchase ratefig, axes = plt.subplots(2, 2, figsize=(14, 10))ax = axes[0][0]ax.hist(df[df['label']==0]['age'], bins=20, alpha=0.6, label='Not Purchased', color='#e74c3c', density=True)ax.hist(df[df['label']==1]['age'], bins=20, alpha=0.6, label='Purchased', color='#2ecc71', density=True)ax.set_title('Age Distribution by Purchase', fontweight='bold')ax.set_xlabel('Age')ax.legend()ax = axes[0][1]gender_rate = df.groupby('gender')['label'].mean()bars = ax.bar(['Female (0)', 'Male (1)'], gender_rate.values, color=['#e91e63', '#2196f3'], edgecolor='white')for bar, v in zip(bars, gender_rate.values):    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.1%}', ha='center', fontweight='bold')ax.set_title('Purchase Rate by Gender', fontweight='bold')ax.set_ylabel('Purchase Rate')ax = axes[1][0]level_rate = df.groupby('user_level')['label'].mean()bars = ax.bar(level_rate.index, level_rate.values, color='#9b59b6', edgecolor='white')for bar, v in zip(bars, level_rate.values):    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.1%}', ha='center', fontweight='bold', fontsize=8)ax.set_title('Purchase Rate by User Level', fontweight='bold')ax.set_xlabel('User Level')ax.set_ylabel('Purchase Rate')ax = axes[1][1]ax.hist(df[df['label']==0]['total_spend'], bins=30, alpha=0.6, label='Not Purchased', color='#e74c3c', density=True)ax.hist(df[df['label']==1]['total_spend'], bins=30, alpha=0.6, label='Purchased', color='#2ecc71', density=True)ax.set_title('Total Spend Distribution by Purchase', fontweight='bold')ax.set_xlabel('Total Spend')ax.legend()plt.suptitle('User Demographics vs Purchase', fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.show()

In [ ]:
# 2.4 Correlation with labelnumeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()corr_matrix = df[numeric_cols].corr()label_corr = corr_matrix['label'].drop('label').sort_values(ascending=False)fig, ax = plt.subplots(figsize=(10, 6))colors_bar = ['#2ecc71' if v > 0 else '#e74c3c' for v in label_corr.values]ax.barh(label_corr.index, label_corr.values, color=colors_bar, edgecolor='white')ax.set_title('Feature Correlation with Purchase (label)', fontsize=13, fontweight='bold')ax.set_xlabel('Pearson Correlation')ax.axvline(x=0, color='black', linewidth=0.5)plt.tight_layout()plt.show()

---## 3. Feature Engineering & Data Preparation

In [ ]:
from sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import LabelEncoder, StandardScalerle = LabelEncoder()df['category_encoded'] = le.fit_transform(df['category'])category_mapping = dict(zip(le.classes_, le.transform(le.classes_)))print('Category mapping:', category_mapping)feature_cols = [c for c in df.columns if c not in ['label', 'category']]X = df[feature_cols]y = df['label']print(f'\nFeature matrix: {X.shape}')print(f'Features: {list(X.columns)}')print(f'Target distribution: {y.value_counts().to_dict()}')X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)print(f'\nTrain: {X_train.shape}, Test: {X_test.shape}')print(f'Train label rate: {y_train.mean():.3f}, Test label rate: {y_test.mean():.3f}')

In [ ]:
scaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_test_scaled = scaler.transform(X_test)

---## 4. Multi-Model Comparison

In [ ]:
from sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifierfrom xgboost import XGBClassifierfrom sklearn.metrics import (    accuracy_score, precision_score, recall_score, f1_score,    roc_auc_score, classification_report, confusion_matrix, roc_curve)models = {    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),    'XGBoost': XGBClassifier(        n_estimators=200, max_depth=6, learning_rate=0.1,        random_state=42, eval_metric='logloss', verbosity=0    )}results = {}for name, model in models.items():    print(f'\n{"="*50}')    print(f'Training: {name}')    print(f'{"="*50}')        if name == 'Logistic Regression':        model.fit(X_train_scaled, y_train)        y_pred = model.predict(X_test_scaled)        y_prob = model.predict_proba(X_test_scaled)[:, 1]    else:        model.fit(X_train, y_train)        y_pred = model.predict(X_test)        y_prob = model.predict_proba(X_test)[:, 1]        acc = accuracy_score(y_test, y_pred)    prec = precision_score(y_test, y_pred)    rec = recall_score(y_test, y_pred)    f1 = f1_score(y_test, y_pred)    auc = roc_auc_score(y_test, y_prob)        results[name] = {        'model': model,        'accuracy': acc,        'precision': prec,        'recall': rec,        'f1': f1,        'auc': auc,        'y_pred': y_pred,        'y_prob': y_prob    }        print(f'Accuracy:  {acc:.4f}')    print(f'Precision: {prec:.4f}')    print(f'Recall:    {rec:.4f}')    print(f'F1 Score:  {f1:.4f}')    print(f'AUC-ROC:   {auc:.4f}')    print(f'\n{classification_report(y_test, y_pred)}')

In [ ]:
comparison_df = pd.DataFrame({    name: {k: v for k, v in res.items() if k in ['accuracy', 'precision', 'recall', 'f1', 'auc']}    for name, res in results.items()}).Tprint('Model Comparison:')print(comparison_df.to_string())best_model_name = comparison_df['auc'].idxmax()print(f'\nBest model by AUC: {best_model_name} (AUC = {comparison_df.loc[best_model_name, "auc"]:.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))ax = axes[0]colors_roc = ['#e74c3c', '#3498db', '#2ecc71']for (name, res), color in zip(results.items(), colors_roc):    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])    ax.plot(fpr, tpr, label=f'{name} (AUC={res["auc"]:.3f})', color=color, linewidth=2)ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)ax.set_xlabel('False Positive Rate')ax.set_ylabel('True Positive Rate')ax.set_title('ROC Curves', fontsize=13, fontweight='bold')ax.legend(loc='lower right')ax = axes[1]metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']x = np.arange(len(metrics))width = 0.25for i, (name, res) in enumerate(results.items()):    vals = [res[m] for m in metrics]    ax.bar(x + i * width, vals, width, label=name, color=colors_roc[i], edgecolor='white')ax.set_xticks(x + width)ax.set_xticklabels([m.upper() for m in metrics])ax.set_ylabel('Score')ax.set_title('Model Metrics Comparison', fontsize=13, fontweight='bold')ax.legend()ax.set_ylim(0, 1.05)plt.tight_layout()plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))for (name, res), ax in zip(results.items(), axes):    cm = confusion_matrix(y_test, res['y_pred'])    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,                xticklabels=['Not Purchased', 'Purchased'],                yticklabels=['Not Purchased', 'Purchased'])    ax.set_title(f'{name}', fontsize=11, fontweight='bold')    ax.set_ylabel('Actual')    ax.set_xlabel('Predicted')plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold', y=1.05)plt.tight_layout()plt.show()

In [ ]:
best_model = results[best_model_name]['model']print(f'Using {best_model_name} for interpretability analysis')

---## 5. Model Interpretability### 5.1 Feature Importance

In [ ]:
from matplotlib.patches import Patchfig, axes = plt.subplots(1, 2, figsize=(16, 6))for ax, model_key in zip(axes, ['Random Forest', 'XGBoost']):    model_obj = results[model_key]['model']    importances = model_obj.feature_importances_    feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=True)        colors_imp = ['#e74c3c' if f in SELLER_CONTROLLABLE else '#3498db' for f in feat_imp.index]    feat_imp.plot(kind='barh', ax=ax, color=colors_imp, edgecolor='white')    ax.set_title(f'{model_key} Feature Importance', fontsize=12, fontweight='bold')    ax.set_xlabel('Importance')legend_elements = [    Patch(facecolor='#e74c3c', label='Seller Controllable'),    Patch(facecolor='#3498db', label='Other Features')]axes[1].legend(handles=legend_elements, loc='lower right')plt.suptitle('Feature Importance (Red = Seller Controllable)', fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.show()

### 5.2 SHAP Analysis

In [ ]:
import shapsample_size = 2000X_sample = X_test.sample(n=sample_size, random_state=42)if best_model_name in ['XGBoost', 'Random Forest']:    explainer = shap.TreeExplainer(best_model)    shap_values = explainer.shap_values(X_sample)else:    X_sample_scaled = scaler.transform(X_sample)    explainer = shap.LinearExplainer(best_model, X_train_scaled)    shap_values = explainer.shap_values(pd.DataFrame(X_sample_scaled, columns=feature_cols))print(f'SHAP values computed for {sample_size} samples')

In [ ]:
plt.figure(figsize=(12, 8))shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, show=False)plt.title('SHAP Feature Impact on Purchase Prediction', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()

In [ ]:
plt.figure(figsize=(10, 6))shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, plot_type='bar', show=False)plt.title('Mean |SHAP| Value (Feature Importance)', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()

In [ ]:
controllable_indices = [feature_cols.index(f) for f in SELLER_CONTROLLABLE if f in feature_cols]shap_controllable = shap_values[:, controllable_indices]X_sample_controllable = X_sample[SELLER_CONTROLLABLE]plt.figure(figsize=(10, 5))shap.summary_plot(shap_controllable, X_sample_controllable, feature_names=SELLER_CONTROLLABLE, show=False)plt.title('SHAP Values: Seller-Controllable Features Only', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()

### 5.3 Partial Dependence Plots (PDP)

In [ ]:
from sklearn.inspection import PartialDependenceDisplaycontrollable_idx = [feature_cols.index(f) for f in SELLER_CONTROLLABLE]tree_model_name = 'XGBoost' if 'XGBoost' in results else 'Random Forest'tree_model = results[tree_model_name]['model']fig, axes = plt.subplots(2, 3, figsize=(18, 10))axes_flat = axes.flatten()for i, (feat_idx, feat_name) in enumerate(zip(controllable_idx, SELLER_CONTROLLABLE)):    PartialDependenceDisplay.from_estimator(        tree_model, X_train, [feat_idx],        feature_names=feature_cols,        ax=axes_flat[i],        kind='average',        line_kw={'color': '#e74c3c', 'linewidth': 2}    )    axes_flat[i].set_title(f'PDP: {feat_name}', fontsize=11, fontweight='bold')plt.suptitle(f'Partial Dependence Plots ({tree_model_name}) - Seller Controllable Features',             fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.show()

### 5.4 Counterfactual AnalysisFor each seller-controllable feature, simulate "what if" changes and measure the predicted purchase probability shift.

In [ ]:
def counterfactual_analysis(model, X_data, feature_cols, controllable_features):    purchased_mask = y == 1    df_purchased = df[purchased_mask]        benchmarks = {}    for cat in df['category'].unique():        cat_purchased = df_purchased[df_purchased['category'] == cat]        benchmarks[cat] = {}        for feat in controllable_features:            if feat == 'has_video':                benchmarks[cat][feat] = cat_purchased[feat].mode().iloc[0] if len(cat_purchased) > 0 else 0            else:                benchmarks[cat][feat] = cat_purchased[feat].median()        return benchmarksbenchmarks = counterfactual_analysis(tree_model, X_test, feature_cols, SELLER_CONTROLLABLE)print('Category-wise Ideal Product Benchmarks (from purchased items):')benchmark_df = pd.DataFrame(benchmarks).Tbenchmark_df.index.name = 'Category'print(benchmark_df.to_string())

In [ ]:
def simulate_counterfactual(model, X_row, feature_name, new_value, feature_cols):    X_original = X_row.values.reshape(1, -1)    original_prob = model.predict_proba(X_original)[0, 1]        X_modified = X_original.copy()    feat_idx = feature_cols.index(feature_name)    X_modified[0, feat_idx] = new_value    new_prob = model.predict_proba(X_modified)[0, 1]        return original_prob, new_prob, new_prob - original_probdef generate_product_recommendations(model, product_features, category_name, benchmarks, feature_cols):    cat_benchmark = benchmarks[category_name]    recommendations = []        for feat in SELLER_CONTROLLABLE:        current_val = product_features[feat]        ideal_val = cat_benchmark[feat]                orig_prob, new_prob, delta = simulate_counterfactual(            model, product_features[feature_cols], feat, ideal_val, feature_cols        )                recommendations.append({            'feature': feat,            'current_value': current_val,            'ideal_value': ideal_val,            'gap': ideal_val - current_val,            'prob_delta': delta,            'original_prob': orig_prob,            'new_prob': new_prob        })        return pd.DataFrame(recommendations).sort_values('prob_delta', ascending=False)# Demo: pick a sample productsample_idx = X_test.index[0]sample_product = df.loc[sample_idx]sample_category = sample_product['category']print(f'Sample Product (Category: {sample_category}):')for feat in SELLER_CONTROLLABLE:    print(f'  {feat}: {sample_product[feat]}')recs = generate_product_recommendations(    tree_model, sample_product, sample_category, benchmarks, feature_cols)print(f'\nCounterfactual Recommendations:')print(f'Current predicted purchase probability: {recs["original_prob"].iloc[0]:.4f}')print(recs[['feature', 'current_value', 'ideal_value', 'gap', 'prob_delta']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))colors_cf = ['#2ecc71' if d > 0 else '#e74c3c' for d in recs['prob_delta']]bars = ax.barh(recs['feature'], recs['prob_delta'], color=colors_cf, edgecolor='white')ax.set_xlabel('Predicted Probability Change')ax.set_title(f'Counterfactual Analysis: Impact of Changing Each Feature to Ideal\n(Category: {sample_category})',             fontsize=12, fontweight='bold')ax.axvline(x=0, color='black', linewidth=0.5)for bar, feat, curr, ideal in zip(bars, recs['feature'], recs['current_value'], recs['ideal_value']):    offset = 0.002 if bar.get_width() >= 0 else -0.002    ha = 'left' if bar.get_width() >= 0 else 'right'    ax.text(bar.get_width() + offset, bar.get_y() + bar.get_height()/2,            f'{curr:.1f} -> {ideal:.1f}', va='center', ha=ha, fontsize=9)plt.tight_layout()plt.show()

---## 6. K-Means User Clustering

In [ ]:
from sklearn.cluster import KMeansfrom sklearn.preprocessing import StandardScaler as SSuser_features_for_cluster = ['age', 'gender', 'user_level', 'purchase_freq', 'total_spend', 'register_days']X_user = df[user_features_for_cluster].copy()scaler_kmeans = SS()X_user_scaled = scaler_kmeans.fit_transform(X_user)inertias = []K_range = range(2, 9)for k in K_range:    km = KMeans(n_clusters=k, random_state=42, n_init=10)    km.fit(X_user_scaled)    inertias.append(km.inertia_)fig, ax = plt.subplots(figsize=(8, 4))ax.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)ax.set_xlabel('Number of Clusters (k)')ax.set_ylabel('Inertia')ax.set_title('Elbow Method for Optimal k', fontsize=13, fontweight='bold')plt.tight_layout()plt.show()

In [ ]:
OPTIMAL_K = 4kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)df['user_cluster'] = kmeans.fit_predict(X_user_scaled)print(f'User cluster distribution:')print(df['user_cluster'].value_counts().sort_index())cluster_profiles = df.groupby('user_cluster')[user_features_for_cluster + ['label']].mean()cluster_profiles['count'] = df.groupby('user_cluster').size()cluster_profiles['purchase_rate'] = df.groupby('user_cluster')['label'].mean()print(f'\nCluster Profiles:')print(cluster_profiles.to_string())

In [ ]:
def name_cluster(row):    traits = []    if row['age'] < 25:        traits.append('Young')    elif row['age'] > 35:        traits.append('Mature')    else:        traits.append('Mid-age')        if row['total_spend'] > df['total_spend'].median():        traits.append('High-spend')    else:        traits.append('Low-spend')        if row['purchase_freq'] > df['purchase_freq'].median():        traits.append('Frequent')    else:        traits.append('Infrequent')        return ' / '.join(traits)cluster_names = {}for idx, row in cluster_profiles.iterrows():    cluster_names[idx] = name_cluster(row)print('Cluster Names:')for k, v in cluster_names.items():    print(f'  Cluster {k}: {v}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))ax = axes[0][0]cluster_pr = df.groupby('user_cluster')['label'].mean()colors_cl = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']cluster_labels = [f'C{i}\n{cluster_names[i]}' for i in range(OPTIMAL_K)]bars = ax.bar(cluster_labels, cluster_pr.values, color=colors_cl[:OPTIMAL_K], edgecolor='white')for bar, v in zip(bars, cluster_pr.values):    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.1%}', ha='center', fontweight='bold')ax.set_title('Purchase Rate by User Cluster', fontweight='bold')ax.set_ylabel('Purchase Rate')ax = axes[0][1]for i in range(OPTIMAL_K):    ax.hist(df[df['user_cluster']==i]['age'], bins=15, alpha=0.5, label=f'C{i}', color=colors_cl[i], density=True)ax.set_title('Age Distribution by Cluster', fontweight='bold')ax.set_xlabel('Age')ax.legend()ax = axes[1][0]cluster_spend = df.groupby('user_cluster')['total_spend'].mean()bars = ax.bar([f'C{i}' for i in range(OPTIMAL_K)], cluster_spend.values, color=colors_cl[:OPTIMAL_K], edgecolor='white')for bar, v in zip(bars, cluster_spend.values):    ax.text(bar.get_x() + bar.get_width()/2, v + 50, f'{v:.0f}', ha='center', fontweight='bold')ax.set_title('Average Total Spend by Cluster', fontweight='bold')ax.set_ylabel('Total Spend')ax = axes[1][1]radar_features = ['age', 'user_level', 'purchase_freq', 'total_spend', 'register_days']cluster_means = df.groupby('user_cluster')[radar_features].mean()cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())cluster_means_norm.plot(kind='bar', ax=ax, color=colors_cl[:OPTIMAL_K], edgecolor='white')ax.set_title('Normalized Feature Means by Cluster', fontweight='bold')ax.set_xlabel('Cluster')ax.legend(fontsize=8, loc='upper right')plt.suptitle('User Cluster Analysis', fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.show()

In [ ]:
coupon_mask = df['coupon_received'] == 1no_coupon_mask = df['coupon_received'] == 0print('Marketing Effectiveness by Cluster:\n')for c in range(OPTIMAL_K):    c_mask = df['user_cluster'] == c    pr_with_coupon = df[c_mask & coupon_mask]['label'].mean() if (c_mask & coupon_mask).sum() > 0 else 0    pr_no_coupon = df[c_mask & no_coupon_mask]['label'].mean() if (c_mask & no_coupon_mask).sum() > 0 else 0        pr_with_video = df[c_mask & (df['has_video']==1)]['label'].mean()    pr_no_video = df[c_mask & (df['has_video']==0)]['label'].mean()        print(f'Cluster {c} ({cluster_names[c]}):')    print(f'  Purchase rate: {df[c_mask]["label"].mean():.1%}')    print(f'  With coupon: {pr_with_coupon:.1%} vs Without: {pr_no_coupon:.1%} (delta: {pr_with_coupon - pr_no_coupon:+.1%})')    print(f'  With video:  {pr_with_video:.1%} vs Without: {pr_no_video:.1%} (delta: {pr_with_video - pr_no_video:+.1%})')    print()

---## 7. Recommendation EngineCombines all analyses into a unified recommendation system.

In [ ]:
def generate_text_recommendations(product_row, category, benchmarks, model, feature_cols, cluster_profiles, cluster_names):    cat_bench = benchmarks[category]    recs = []        X_input = product_row[feature_cols].values.reshape(1, -1)    current_prob = model.predict_proba(X_input)[0, 1]        # --- Title Optimization ---    tl = product_row['title_length']    tl_ideal = cat_bench['title_length']    if tl < tl_ideal * 0.8:        _, new_p, delta = simulate_counterfactual(model, product_row[feature_cols], 'title_length', tl_ideal, feature_cols)        recs.append({            'dimension': 'Title Optimization',            'issue': f'Title too short ({tl:.0f} chars)',            'suggestion': f'Increase title length to ~{tl_ideal:.0f} chars with more product keywords',            'expected_lift': delta,            'priority': abs(delta)        })    elif tl > tl_ideal * 1.5:        _, new_p, delta = simulate_counterfactual(model, product_row[feature_cols], 'title_length', tl_ideal, feature_cols)        recs.append({            'dimension': 'Title Optimization',            'issue': f'Title too long ({tl:.0f} chars)',            'suggestion': f'Shorten title to ~{tl_ideal:.0f} chars, keep it concise',            'expected_lift': delta,            'priority': abs(delta)        })        emo = product_row['title_emo_score']    emo_ideal = cat_bench['title_emo_score']    if emo < emo_ideal * 0.7:        _, new_p, delta = simulate_counterfactual(model, product_row[feature_cols], 'title_emo_score', emo_ideal, feature_cols)        recs.append({            'dimension': 'Title Optimization',            'issue': f'Title emotion score low ({emo:.2f})',            'suggestion': f'Use more engaging language, target emotion score ~{emo_ideal:.2f}',            'expected_lift': delta,            'priority': abs(delta)        })        # --- Content Strategy ---    if product_row['has_video'] == 0:        _, new_p, delta = simulate_counterfactual(model, product_row[feature_cols], 'has_video', 1, feature_cols)        recs.append({            'dimension': 'Content Strategy',            'issue': 'No product video',            'suggestion': 'Add a product demonstration video',            'expected_lift': delta,            'priority': abs(delta)        })        img = product_row['img_count']    img_ideal = cat_bench['img_count']    if img < img_ideal * 0.7:        _, new_p, delta = simulate_counterfactual(model, product_row[feature_cols], 'img_count', img_ideal, feature_cols)        recs.append({            'dimension': 'Content Strategy',            'issue': f'Too few images ({img:.0f})',            'suggestion': f'Add more product images, target ~{img_ideal:.0f} images',            'expected_lift': delta,            'priority': abs(delta)        })        # --- Pricing Strategy ---    price = product_row['price']    price_ideal = cat_bench['price']    discount = product_row['discount_rate']    discount_ideal = cat_bench['discount_rate']        if price > price_ideal * 1.3:        _, new_p, delta = simulate_counterfactual(model, product_row[feature_cols], 'price', price_ideal, feature_cols)        recs.append({            'dimension': 'Pricing Strategy',            'issue': f'Price above category median ({price:.1f} vs {price_ideal:.1f})',            'suggestion': f'Consider price reduction or bundle offers. Category median: {price_ideal:.1f}',            'expected_lift': delta,            'priority': abs(delta)        })        if discount < discount_ideal * 0.5 and discount_ideal > 0:        _, new_p, delta = simulate_counterfactual(model, product_row[feature_cols], 'discount_rate', discount_ideal, feature_cols)        recs.append({            'dimension': 'Pricing Strategy',            'issue': f'Discount rate too low ({discount:.1%} vs category avg {discount_ideal:.1%})',            'suggestion': f'Consider offering ~{discount_ideal:.0%} discount',            'expected_lift': delta,            'priority': abs(delta)        })        # --- Target Audience ---    cat_cluster_rates = df[df['category'] == category].groupby('user_cluster')['label'].mean()    best_cluster = cat_cluster_rates.idxmax()    best_cluster_rate = cat_cluster_rates.max()        recs.append({        'dimension': 'Target Audience',        'issue': 'Audience targeting',        'suggestion': f'Focus on Cluster {best_cluster} ({cluster_names[best_cluster]}) - purchase rate {best_cluster_rate:.1%} for {category}',        'expected_lift': 0,        'priority': 0.5    })        recs_df = pd.DataFrame(recs).sort_values('priority', ascending=False)    return current_prob, recs_df# Demosample_idx = X_test.index[5]sample_product = df.loc[sample_idx]sample_category = sample_product['category']current_prob, recs_df = generate_text_recommendations(    sample_product, sample_category, benchmarks, tree_model, feature_cols, cluster_profiles, cluster_names)print(f'Product Analysis (Category: {sample_category})')print(f'Current purchase probability: {current_prob:.2%}')print(f'\nRecommendations (ranked by expected impact):')print('=' * 80)for _, rec in recs_df.iterrows():    lift_str = f'{rec["expected_lift"]:+.2%}' if rec['expected_lift'] != 0 else 'N/A'    print(f'\n[{rec["dimension"]}]')    print(f'  Issue:      {rec["issue"]}')    print(f'  Suggestion: {rec["suggestion"]}')    print(f'  Expected purchase rate lift: {lift_str}')

---## 8. Save Artifacts for Streamlit App

In [ ]:
import pickleartifacts = {    'model': tree_model,    'model_name': tree_model_name,    'feature_cols': feature_cols,    'scaler': scaler,    'label_encoder': le,    'category_mapping': category_mapping,    'benchmarks': benchmarks,    'kmeans': kmeans,    'scaler_kmeans': scaler_kmeans,    'user_features_for_cluster': user_features_for_cluster,    'cluster_profiles': cluster_profiles,    'cluster_names': cluster_names,    'SELLER_CONTROLLABLE': SELLER_CONTROLLABLE,    'comparison_df': comparison_df,    'OPTIMAL_K': OPTIMAL_K,}with open('model_artifacts.pkl', 'wb') as f:    pickle.dump(artifacts, f)# Also save the processed dataframe for the Streamlit appdf.to_csv('processed_data.csv', index=False)print('Artifacts saved to model_artifacts.pkl')print('Processed data saved to processed_data.csv')print(f'Keys: {list(artifacts.keys())}')

---## Summary### Key Findings1. **Model Performance**: Multi-model comparison with LR, RF, and XGBoost. The best model is selected by AUC-ROC.2. **Feature Importance**: SHAP analysis reveals which seller-controllable features have the most impact on purchase probability.3. **Partial Dependence**: PDP plots show the marginal effect of each controllable feature on predicted purchase probability.4. **Counterfactual Analysis**: For each product, we quantify the expected purchase rate lift from specific feature changes.5. **User Clustering**: K-Means identifies distinct user segments with different purchase behaviors and marketing responsiveness.### Actionable Recommendations Framework- **Title**: Optimize length and emotional tone based on category benchmarks- **Content**: Add video and sufficient product images- **Pricing**: Align with category norms, consider strategic discounts- **Targeting**: Focus on user clusters with highest purchase affinity for the product category